In [8]:
import pandas as pd
import urllib
from sqlalchemy import create_engine, text

# 1. Load data and configure connection parameters
df = pd.read_csv('../data/processed/daily_port_cargo_generation.csv')
server = 'LOQ\SQLEXPRESS'
database = 'TradeFlow'

conn_str = f"Driver={{ODBC Driver 17 for SQL Server}};Server={server};Database={database};Trusted_Connection=yes;"
engine = create_engine(f"mssql+pyodbc:///?odbc_connect={urllib.parse.quote_plus(conn_str)}")

# 2. Schema establishment and table truncation block
table_ddl = """
IF NOT EXISTS (SELECT * FROM sys.objects WHERE object_id = OBJECT_ID(N'[dbo].[daily_port_cargo]') AND type in (N'U'))
BEGIN
    CREATE TABLE [dbo].[daily_port_cargo] (
        [Record_Date] DATE NOT NULL,
        [Year] INT NOT NULL,
        [Month] INT NOT NULL,
        [Day] INT NOT NULL,
        [Port_Region] VARCHAR(50) NOT NULL,
        [Cargo_Type] VARCHAR(50) NOT NULL,
        [Import_Volume_MMT] DECIMAL(10,3) NOT NULL,
        [Export_Volume_MMT] DECIMAL(10,3) NOT NULL,
        [Total_Volume_MMT] DECIMAL(10,3) NOT NULL,
        [Vessel_Arrival_Count] INT NOT NULL,
        [Avg_Berth_Turnaround_Days] DECIMAL(10,2) NOT NULL,
        CONSTRAINT PK_PortOperations PRIMARY KEY CLUSTERED ([Record_Date] ASC, [Cargo_Type] ASC)
    );
END
ELSE
BEGIN
    TRUNCATE TABLE [dbo].[daily_port_cargo];
END
"""

with engine.connect() as conn:
    conn.execute(text(table_ddl))
    conn.commit()

# 3. Bulk append operation
df.to_sql(
    name='daily_port_cargo',
    con=engine,
    if_exists='append',
    index=False,
    chunksize=500
)

print(f"Data migration complete. Successfully inserted {len(df)} rows into {database}.")

<>:7: SyntaxWarning: "\S" is an invalid escape sequence. Such sequences will not work in the future. Did you mean "\\S"? A raw string is also an option.
<>:7: SyntaxWarning: "\S" is an invalid escape sequence. Such sequences will not work in the future. Did you mean "\\S"? A raw string is also an option.
C:\Users\offic\AppData\Local\Temp\ipykernel_12072\1637170554.py:7: SyntaxWarning: "\S" is an invalid escape sequence. Such sequences will not work in the future. Did you mean "\\S"? A raw string is also an option.
  server = 'LOQ\SQLEXPRESS'


Data migration complete. Successfully inserted 3291 rows into TradeFlow.
